# Seeded plasma-column full-transport runs and diagnostics

This notebook runs the seeded neutralization cases with minimal live output. WarpX stdout/stderr is redirected to log files so JupyterLab does not freeze from very large cell output.

In [1]:
from pathlib import Path
import os
import sys
import subprocess
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

WORK = Path.home() / "Work" / "simulation_codes-working"
WARPX_DATA_DIR = WORK / "warpx-data"

# Use the v7 script with safe ratio post-processing and reducedfiles/ path.
SCRIPT = Path("plasma_column_mcc_picmi_v7.py").resolve()

os.environ["WARPX_DATA_DIR"] = str(WARPX_DATA_DIR)
os.environ["LD_LIBRARY_PATH"] = (
    "/home/cspark/Work/simulation_codes-working/warpx/install/lib:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)

print("Python:", sys.executable)
print("Script:", SCRIPT)
print("WarpX data:", WARPX_DATA_DIR)


Python: /home/cspark/Work/simulation_codes-working/miniforge3/envs/warpx-dev/bin/python
Script: /home/cspark/Work/projects/plasma_column/simulations/plasma_column_mcc_picmi_v7.py
WarpX data: /home/cspark/Work/simulation_codes-working/warpx-data


## 1. Run configuration

For a first full transport pass, run one case at a time. Start with `h2_seeded`, then `kr_seeded`, then `vacuum`.

In [2]:
# Full-domain transport needs roughly 1e5-1.2e5 steps for the current 0.26 m z-domain.
# Diagnostics are deliberately sparse to keep output manageable.
common_args = [
    "--warpx_data_dir", str(WARPX_DATA_DIR),
    "--pressure_torr", "1e-5",
    "--plasma_age", "2e-4",
    "--max_steps", "120000",
    "--diag_period", "5000",
    "--reduced_diag_period", "100",
    "--reduced_diag_dir", "reducedfiles/",
    "--nx", "24", "--ny", "24", "--nz", "128",
]

cases = [
    {
        "name": "h2_seeded_full",
        "args": ["--gas", "H2", "--neutralization", "-1", "--mcc", "none"],
    },
    {
        "name": "kr_seeded_full",
        "args": ["--gas", "Kr", "--neutralization", "-1", "--mcc", "none"],
    },
    {
        "name": "vacuum_full",
        "args": ["--gas", "H2", "--neutralization", "0.0", "--mcc", "none"],
    },
]

# For testing this notebook without a long run, change max_steps above to 2000.
cases


[{'name': 'h2_seeded_full',
  'args': ['--gas', 'H2', '--neutralization', '-1', '--mcc', 'none']},
 {'name': 'kr_seeded_full',
  'args': ['--gas', 'Kr', '--neutralization', '-1', '--mcc', 'none']},
 {'name': 'vacuum_full',
  'args': ['--gas', 'H2', '--neutralization', '0.0', '--mcc', 'none']}]

## 2. Helper functions

In [3]:
def run_case(case):
    out_dir = Path("runs") / case["name"]
    out_dir.mkdir(parents=True, exist_ok=True)
    log_dir = Path("logs")
    log_dir.mkdir(exist_ok=True)
    log_file = log_dir / f"{case['name']}.log"

    cmd = [
        sys.executable, str(SCRIPT),
        "--run",
        "--output_dir", str(out_dir),
    ] + common_args + case["args"]

    print("=" * 90)
    print("Running:", case["name"])
    print("Output:", out_dir)
    print("Log:", log_file)
    print("Command:")
    print(" ".join(cmd))
    print("=" * 90)

    t0 = time.time()
    with log_file.open("w") as lf:
        result = subprocess.run(
            cmd,
            stdout=lf,
            stderr=subprocess.STDOUT,
            env=os.environ.copy(),
            text=True,
        )

    elapsed = time.time() - t0
    print(f"Return code: {result.returncode}")
    print(f"Elapsed: {elapsed/60:.2f} min")

    # Print only the last part of the log to avoid overwhelming Jupyter.
    if log_file.exists():
        tail_lines = log_file.read_text(errors="replace").splitlines()[-40:]
        print("\n--- log tail ---")
        print("\n".join(tail_lines))

    if result.returncode != 0:
        raise RuntimeError(f"Case {case['name']} failed. See {log_file}")

    return out_dir, log_file


def check_case_outputs(case_name):
    out_dir = Path("runs") / case_name
    targets = [
        out_dir / "neutralization_from_particle_number.csv",
        out_dir / "neutralization_model.csv",
        out_dir / "reducedfiles" / "particle_number.txt",
        out_dir / "reducedfiles" / "timestep.txt",
    ]
    print("\nCASE:", case_name)
    ok = True
    for p in targets:
        exists = p.exists()
        print(f"  {p}: {'OK' if exists else 'MISSING'}")
        ok = ok and exists
    return ok


def load_history(case_name):
    out_dir = Path("runs") / case_name
    hist = pd.read_csv(out_dir / "neutralization_from_particle_number.csv")
    model = pd.read_csv(out_dir / "neutralization_model.csv")
    return hist, model


## 3. Run one case

Run one full case first. After it completes and the plots look reasonable, run the next cases.

In [4]:
# Recommended first full case:
out_dir, log_file = run_case(cases[0])  # h2_seeded_full
check_case_outputs(cases[0]["name"])


Running: h2_seeded_full
Output: runs/h2_seeded_full
Log: logs/h2_seeded_full.log
Command:
/home/cspark/Work/simulation_codes-working/miniforge3/envs/warpx-dev/bin/python /home/cspark/Work/projects/plasma_column/simulations/plasma_column_mcc_picmi_v7.py --run --output_dir runs/h2_seeded_full --warpx_data_dir /home/cspark/Work/simulation_codes-working/warpx-data --pressure_torr 1e-5 --plasma_age 2e-4 --max_steps 120000 --diag_period 5000 --reduced_diag_period 100 --reduced_diag_dir reducedfiles/ --nx 24 --ny 24 --nz 128 --gas H2 --neutralization -1 --mcc none
Return code: 0
Elapsed: 1924.28 min

--- log tail ---
WarpX::EvolveE()                                                       120000      291.1      291.1      291.1   0.25%
amrex::unpackBuffer                                                    480000      254.5      254.5      254.5   0.22%
Other                                                                31508967      982.6      982.6      982.6   0.85%
-------------------------

True

In [5]:
# Then run Kr:
out_dir, log_file = run_case(cases[1])  # kr_seeded_full
check_case_outputs(cases[1]["name"])


Running: kr_seeded_full
Output: runs/kr_seeded_full
Log: logs/kr_seeded_full.log
Command:
/home/cspark/Work/simulation_codes-working/miniforge3/envs/warpx-dev/bin/python /home/cspark/Work/projects/plasma_column/simulations/plasma_column_mcc_picmi_v7.py --run --output_dir runs/kr_seeded_full --warpx_data_dir /home/cspark/Work/simulation_codes-working/warpx-data --pressure_torr 1e-5 --plasma_age 2e-4 --max_steps 120000 --diag_period 5000 --reduced_diag_period 100 --reduced_diag_dir reducedfiles/ --nx 24 --ny 24 --nz 128 --gas Kr --neutralization -1 --mcc none
Return code: 0
Elapsed: 2111.50 min

--- log tail ---
FabArray::FillBoundary()                                               727353      341.1      341.1      341.1   0.27%
WarpX::DampPML()                                                       120000      289.9      289.9      289.9   0.23%
Other                                                                31393391      980.5      980.5      980.5   0.77%
-------------------------

True

In [4]:
# Then run vacuum:
out_dir, log_file = run_case(cases[2])  # vacuum_full
check_case_outputs(cases[2]["name"])


Running: vacuum_full
Output: runs/vacuum_full
Log: logs/vacuum_full.log
Command:
/home/cspark/Work/simulation_codes-working/miniforge3/envs/warpx-dev/bin/python /home/cspark/Work/projects/plasma_column/simulations/plasma_column_mcc_picmi_v7.py --run --output_dir runs/vacuum_full --warpx_data_dir /home/cspark/Work/simulation_codes-working/warpx-data --pressure_torr 1e-5 --plasma_age 2e-4 --max_steps 120000 --diag_period 5000 --reduced_diag_period 100 --reduced_diag_dir reducedfiles/ --nx 24 --ny 24 --nz 128 --gas H2 --neutralization 0.0 --mcc none
Return code: 0
Elapsed: 1305.73 min

--- log tail ---
FillBoundaryAndSync_nowait()                                          2880000      157.5      157.5      157.5   0.20%
FabArray::FillBoundaryAndSync()                                       1440000      146.6      146.6      146.6   0.19%
Other                                                                37516187      713.4      713.4      713.4   0.91%
------------------------------------

True

## 4. Plot neutralization and species histories

In [ ]:
# Choose whichever cases have completed.
completed_cases = [c["name"] for c in cases if check_case_outputs(c["name"])]
completed_cases


In [ ]:
histories = {}
models = {}

for name in completed_cases:
    hist, model = load_history(name)
    histories[name] = hist
    models[name] = model

    print("\n", name)
    display(hist.head())
    display(hist.tail())


In [ ]:
# Plot global neutralization from particle counts against seeded analytic model.
plt.figure(figsize=(9, 5))
for name, hist in histories.items():
    if "global_net_neutralization" in hist.columns:
        plt.plot(
            hist["time_s"],
            hist["global_net_neutralization"],
            label=f"{name}: global (Ne-Ni)/Np", linewidth=4
        )
for name, model in models.items():
    plt.plot(
        model["t_est_s"],
        model["neutralization_fraction_model"],
        "--",
        label=f"{name}: seeded model", linewidth=4
    )

plt.xlabel("time [s]", fontsize=16)
plt.ylabel("neutralization fraction", fontsize=16)
plt.title("Global neutralization history", fontsize=16)
plt.grid(True)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Plot physical particle populations for each completed case.
for name, hist in histories.items():
    plt.figure(figsize=(9, 5))
    plt.plot(hist["time_s"], hist["physical_beam_protons"], label="beam protons", linewidth=4)
    plt.plot(hist["time_s"], hist["physical_plasma_electrons"], label="plasma electrons", linewidth=4)
    plt.plot(hist["time_s"], hist["physical_gas_ions"], label="gas ions", linewidth=4)
    plt.xlabel("time [s]", fontsize=16)
    plt.ylabel("global physical particle count", fontsize=16)
    plt.title(f"Species populations: {name}", fontsize=16)
    plt.grid(True)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# Plot ratios only where valid; NaN rows are automatically skipped by matplotlib.
for name, hist in histories.items():
    plt.figure(figsize=(9, 5))
    if "electron_over_proton" in hist.columns:
        plt.plot(hist["time_s"], hist["electron_over_proton"], label="Ne/Np", linewidth=4)
    if "ion_over_proton" in hist.columns:
        plt.plot(hist["time_s"], hist["ion_over_proton"], label="Ni/Np", linewidth=4)
    if "global_net_neutralization" in hist.columns:
        plt.plot(hist["time_s"], hist["global_net_neutralization"], label="(Ne-Ni)/Np", linewidth=4)
    plt.xlabel("time [s]", fontsize=16)
    plt.ylabel("ratio", fontsize=16)
    plt.title(f"Global particle ratios: {name}", fontsize=16)
    plt.grid(True)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.legend()
    plt.tight_layout()
    plt.show()


## 5. Estimate beam transit coverage

This checks whether the simulated time covers the proton transit through the z-domain.

In [ ]:
# Approximate 30 keV proton speed from previous dry runs.
v_proton = 2.397295197e6  # m/s
z_length = 0.24 - (-0.02)
t_transit = z_length / v_proton

for name, hist in histories.items():
    t_final = hist["time_s"].dropna().iloc[-1]
    print(f"{name:20s} final time = {t_final:.3e} s, "
          f"transit fraction = {t_final/t_transit:.2f}, "
          f"distance traveled ≈ {v_proton*t_final:.3f} m")
print(f"Estimated full-domain transit time: {t_transit:.3e} s")


## 6. Optional: save summary plots

In [ ]:
plot_dir = Path("plots")
plot_dir.mkdir(exist_ok=True)

# Save neutralization comparison.
plt.figure(figsize=(9, 5))
for name, hist in histories.items():
    if "global_net_neutralization" in hist.columns:
        plt.plot(hist["time_s"], hist["global_net_neutralization"], label=f"{name}: global", linewidth=4)
for name, model in models.items():
    plt.plot(model["t_est_s"], model["neutralization_fraction_model"], "--", label=f"{name}: model", linewidth=4)
plt.xlabel("time [s]", fontsize=16)
plt.ylabel("neutralization fraction", fontsize=16)
plt.title("Seeded neutralization comparison", fontsize=16)
plt.grid(True)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.legend()
plt.tight_layout()
plt.savefig(plot_dir / "seeded_neutralization_comparison.png", dpi=200)
plt.show()

print("Saved:", plot_dir / "seeded_neutralization_comparison.png")
